## EXPLORATION GOLD DATASET

In [1]:
#import pandas as pd
#import sqlalchemy as sa
#from indusense.db.session import create_postgres_engine
#from indusense.db.models import GoldMachineHourlyFeature

#engine = create_postgres_engine()
#stmt = sa.select(GoldMachineHourlyFeature)

#gold_df = pd.read_sql(stmt, engine)
#gold_df.head()

###############

import pandas as pd

DATA_PATH = "C:\\Formation\\gold_dataset\\gold_dataset_20260611-155332.csv"

COLS_TO_DROP = [
    'machine_id_std',
    'future_incident_count_6h',
    'future_incident_count_12h',
    'future_incident_count_24h',
    'future_incident_count_48h',
]

gold_df = pd.read_csv(DATA_PATH)
gold_df = gold_df.drop(columns=COLS_TO_DROP)

print(f"Dimensions : {gold_df.shape}")
print(gold_df.head())

print(gold_df.isna().sum())
print(gold_df[gold_df.isna().any(axis=1)].head())

Dimensions : (134280, 85)
          window_start  temp_mean_1h  temp_max_1h  pressure_mean_1h  \
0  2025-06-01 00:00:00         45.44        45.44           194.302   
1  2025-06-01 01:00:00         47.87        47.87           194.391   
2  2025-06-01 02:00:00         50.46        50.46           195.641   
3  2025-06-01 03:00:00         48.62        48.62           197.737   
4  2025-06-01 04:00:00         51.09        51.09           196.253   

   pressure_max_1h  voltage_mean_1h  voltage_max_1h  rotation_mean_1h  \
0          194.302           227.57          227.57            1441.7   
1          194.391           227.48          227.48            1437.8   
2          195.641           228.68          228.68            1484.6   
3          197.737           228.44          228.44            1488.9   
4          196.253           227.84          227.84            1448.6   

   rotation_max_1h  pieces_produced_sum_1h  ...  \
0           1441.7                       4  ...   
1     

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [3]:
# ===== SETUP MLFLOW =====
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from contextlib import nullcontext

# ── Flag ──────────────────────────────────────────────────────────────
USE_MLFLOW = False   # False → toutes les cellules d'entraînement tournent sans MLflow
# ──────────────────────────────────────────────────────────────────────

MLFLOW_URI = "file:C:/indusense/mlruns"

if USE_MLFLOW:
    mlflow.set_tracking_uri(MLFLOW_URI)
    mlflow.set_experiment("indusense-failure-prediction")
    print(f"Tracking URI : {mlflow.get_tracking_uri()}")
    print("Experiment   : indusense-failure-prediction")
    print(f"\nPour lancer l'UI :")
    print(f"  mlflow ui --backend-store-uri {MLFLOW_URI}")
else:
    print("MLflow désactivé (USE_MLFLOW = False)")

# --- Wrappers ---------------------------------------------------------
def mlflow_run(run_name):
    return mlflow.start_run(run_name=run_name) if USE_MLFLOW else nullcontext()

def mlf_log_params(params):
    if USE_MLFLOW: mlflow.log_params(params)

def mlf_log_param(key, value):
    if USE_MLFLOW: mlflow.log_param(key, value)

def mlf_log_metric(key, value):
    if USE_MLFLOW: mlflow.log_metric(key, value)

def mlf_log_sklearn(model, name="model"):
    if USE_MLFLOW: mlflow.sklearn.log_model(model, name)

def mlf_log_xgboost(model, name="model"):
    if USE_MLFLOW: mlflow.xgboost.log_model(model, name)

MLflow désactivé (USE_MLFLOW = False)


## Recommandation pour action sur les NaN
Si une colonne est critique et que les valeurs manquantes sont rares, dropna() est souvent acceptable.
Si les valeurs manquantes sont fréquentes, mieux vaut fillna() avec une moyenne/médiane ou une méthode d’interpolation.
Toujours commencer par inspecter.

En résumé : inspecter les NaN, puis choisir dropna, fillna, ou interpolate selon si vous pouvez supprimer les lignes ou si vous devez conserver et estimer les valeurs manquantes.

Supprimer les lignes contenant des NaN
gold_df_clean = gold_df.dropna()

Remplacer les NaN par une valeur fixe
gold_df_filled = gold_df.fillna(0)

Interpolation si les données sont temporelles / ordonnées
gold_df = gold_df.interpolate()

In [4]:
# Résumé concis des NaN
nan_summary = gold_df.isna().sum()
cols_with_nan = nan_summary[nan_summary > 0]

print(f"Dimensions: {gold_df.shape}")
print(f"\nColonnes avec NaN: {len(cols_with_nan)}")
if len(cols_with_nan) > 0:
    print(cols_with_nan)
    print(f"\nPourcentage de NaN (max): {(cols_with_nan.max() / len(gold_df) * 100):.2f}%")
else:
    print("✓ Aucun NaN dans le dataset!")
    
nan_pct = (gold_df.isna().sum() / len(gold_df) * 100)
nan_pct[nan_pct > 0].sort_values(ascending=False)

Dimensions: (134280, 85)

Colonnes avec NaN: 21
temp_std_6h                           15
pressure_std_6h                       15
voltage_std_6h                        15
rotation_std_6h                       15
temp_std_12h                          15
pressure_std_12h                      15
voltage_std_12h                       15
rotation_std_12h                      15
temp_std_24h                          15
pressure_std_24h                      15
voltage_std_24h                       15
rotation_std_24h                      15
temp_trend_6h                         90
pressure_trend_6h                     90
voltage_trend_6h                      90
rotation_trend_6h                     90
temp_zscore_24h                       15
incident_max_severity_1h          133386
incident_max_severity_prev_24h    114532
hours_since_last_incident           3102
days_since_last_maintenance         5192
dtype: int64

Pourcentage de NaN (max): 99.33%


incident_max_severity_1h          99.334227
incident_max_severity_prev_24h    85.293417
days_since_last_maintenance        3.866548
hours_since_last_incident          2.310098
temp_trend_6h                      0.067024
rotation_trend_6h                  0.067024
pressure_trend_6h                  0.067024
voltage_trend_6h                   0.067024
temp_std_12h                       0.011171
temp_std_6h                        0.011171
rotation_std_6h                    0.011171
voltage_std_6h                     0.011171
pressure_std_6h                    0.011171
rotation_std_24h                   0.011171
voltage_std_24h                    0.011171
pressure_std_24h                   0.011171
temp_std_24h                       0.011171
voltage_std_12h                    0.011171
rotation_std_12h                   0.011171
pressure_std_12h                   0.011171
temp_zscore_24h                    0.011171
dtype: float64

Stratégie recommandée par type de colonne
1. Supprimer complètement ambient_humidity_pct (100% de NaN)
2. Pour incident_max_severity_prev_24h (95% de NaN) → Supprimer la colonne ou remplacer par 0 (pas d'incident)
3. Pour hours_since_last_incident (13% de NaN) → Remplacer par la médiane ou -1 (pas d'incident connu)
4. Pour les colonnes pressure_* et temp_* (1-2% de NaN) → Interpolation (données temporelles)


In [5]:
# ===== MODULE DE NETTOYAGE DES NaN =====

print("=== NETTOYAGE DES NaN ===\n")

# 1. Supprimer les colonnes avec 100% de NaN
cols_to_drop = nan_pct[nan_pct == 100].index.tolist()
if cols_to_drop:
    print(f"1. Suppression de colonnes vides (100% NaN): {cols_to_drop}")
    gold_df = gold_df.drop(columns=cols_to_drop)
else:
    print("1. Aucune colonne avec 100% de NaN")

# 2. Traiter les colonnes avec > 50% de NaN
cols_high_nan = nan_pct[(nan_pct > 50) & (nan_pct < 100)].index.tolist()
if cols_high_nan:
    print(f"\n2. Colonnes avec >50% NaN: {cols_high_nan}")
    for col in cols_high_nan:
        print(f"   - {col}: remplissage avec -1")
        gold_df[col] = gold_df[col].fillna(-1)

# 3. Interpolation pour colonnes temporelles (pressure, temp)
cols_to_interpolate = [col for col in gold_df.columns 
                       if any(x in col for x in ['pressure', 'temp', 'trend'])]
cols_to_interpolate = [col for col in cols_to_interpolate if gold_df[col].isna().sum() > 0]
if cols_to_interpolate:
    print(f"\n3. Interpolation pour colonnes temporelles: {cols_to_interpolate}")
    for col in cols_to_interpolate:
        gold_df[col] = gold_df[col].interpolate(method='linear', limit_direction='both')

# 4. Remplissage avec médiane pour colonnes restantes
remaining_nan = gold_df.columns[gold_df.isna().any()].tolist()
if remaining_nan:
    print(f"\n4. Remplissage avec médiane: {remaining_nan}")
    for col in remaining_nan:
        median_val = gold_df[col].median()
        gold_df[col] = gold_df[col].fillna(median_val)
        print(f"   - {col}: {gold_df[col].isna().sum()} NaN restants")

# 5. Vérification finale
total_nan = gold_df.isna().sum().sum()
print(f"\n✓ RÉSULTAT FINAL: {total_nan} NaN restants")
print(f"Dimensions: {gold_df.shape}")


=== NETTOYAGE DES NaN ===

1. Aucune colonne avec 100% de NaN

2. Colonnes avec >50% NaN: ['incident_max_severity_1h', 'incident_max_severity_prev_24h']
   - incident_max_severity_1h: remplissage avec -1
   - incident_max_severity_prev_24h: remplissage avec -1

3. Interpolation pour colonnes temporelles: ['temp_std_6h', 'pressure_std_6h', 'temp_std_12h', 'pressure_std_12h', 'temp_std_24h', 'pressure_std_24h', 'temp_trend_6h', 'pressure_trend_6h', 'voltage_trend_6h', 'rotation_trend_6h', 'temp_zscore_24h']

4. Remplissage avec médiane: ['voltage_std_6h', 'rotation_std_6h', 'voltage_std_12h', 'rotation_std_12h', 'voltage_std_24h', 'rotation_std_24h', 'hours_since_last_incident', 'days_since_last_maintenance']
   - voltage_std_6h: 0 NaN restants
   - rotation_std_6h: 0 NaN restants
   - voltage_std_12h: 0 NaN restants
   - rotation_std_12h: 0 NaN restants
   - voltage_std_24h: 0 NaN restants
   - rotation_std_24h: 0 NaN restants
   - hours_since_last_incident: 0 NaN restants
   - days_sin

In [6]:
# ===== ANALYSE CORRELATION ENTRE PRESSURE ET NaN =====

# Recharger le dataset original pour avoir les NaN
gold_df_original = pd.read_csv(DATA_PATH)

print("=== CORRÉLATION PRESSURE & NaN ===\n")

# 1. Identifier les colonnes pressure avec NaN
pressure_cols = [col for col in gold_df_original.columns if 'pressure' in col]
pressure_cols_with_nan = [col for col in pressure_cols if gold_df_original[col].isna().sum() > 0]

print(f"Colonnes pressure avec NaN: {pressure_cols_with_nan}\n")

# 2. Créer une colonne indicatrice: 1 si NaN dans pressure, 0 sinon
gold_df_original['has_pressure_nan'] = gold_df_original[pressure_cols_with_nan].isna().any(axis=1).astype(int)

# 3. Analyser les patterns des NaN
print(f"Nombre de lignes avec NaN dans pressure: {gold_df_original['has_pressure_nan'].sum()}")
print(f"Pourcentage: {gold_df_original['has_pressure_nan'].sum() / len(gold_df_original) * 100:.2f}%\n")

# 4. Vérifier si les NaN pressure coincident avec NaN d'autres colonnes
print("Corrélation entre NaN pressure et NaN d'autres colonnes:")
cols_nan_correlation = []
for col in gold_df_original.columns:
    if col not in pressure_cols and gold_df_original[col].isna().sum() > 0:
        both_nan = ((gold_df_original[col].isna() & gold_df_original['has_pressure_nan'].astype(bool)).sum())
        if both_nan > 0:
            pct = both_nan / gold_df_original['has_pressure_nan'].sum() * 100
            cols_nan_correlation.append((col, both_nan, pct))

if cols_nan_correlation:
    for col, count, pct in sorted(cols_nan_correlation, key=lambda x: x[2], reverse=True):
        print(f"  {col}: {count} lignes ({pct:.1f}% des lignes avec pressure NaN)")
else:
    print("  Aucune corrélation détectée - NaN pressure sont indépendants")

# 5. Vérifier si c'est un pattern temporel (lignes consécutives)
nan_indices = gold_df_original[gold_df_original['has_pressure_nan'] > 0].index.values
if len(nan_indices) > 0:
    print(f"\nPattern temporel des NaN pressure:")
    print(f"  Première occurrence: index {nan_indices[0]}")
    print(f"  Dernière occurrence: index {nan_indices[-1]}")
    gaps = nan_indices[1:] - nan_indices[:-1]
    print(f"  Écart moyen entre NaN: {gaps.mean():.0f} lignes")
    print(f"  Écart max: {gaps.max()} lignes")

=== CORRÉLATION PRESSURE & NaN ===

Colonnes pressure avec NaN: ['pressure_std_6h', 'pressure_std_12h', 'pressure_std_24h', 'pressure_trend_6h']

Nombre de lignes avec NaN dans pressure: 90
Pourcentage: 0.07%

Corrélation entre NaN pressure et NaN d'autres colonnes:
  temp_trend_6h: 90 lignes (100.0% des lignes avec pressure NaN)
  voltage_trend_6h: 90 lignes (100.0% des lignes avec pressure NaN)
  rotation_trend_6h: 90 lignes (100.0% des lignes avec pressure NaN)
  days_since_last_maintenance: 90 lignes (100.0% des lignes avec pressure NaN)
  incident_max_severity_1h: 89 lignes (98.9% des lignes avec pressure NaN)
  incident_max_severity_prev_24h: 89 lignes (98.9% des lignes avec pressure NaN)
  hours_since_last_incident: 89 lignes (98.9% des lignes avec pressure NaN)
  temp_std_6h: 15 lignes (16.7% des lignes avec pressure NaN)
  voltage_std_6h: 15 lignes (16.7% des lignes avec pressure NaN)
  rotation_std_6h: 15 lignes (16.7% des lignes avec pressure NaN)
  temp_std_12h: 15 lignes (

# ===== SÉPARATION TRAIN / TEST =====

# Colonnes à exclure des features
COLS_META = ['machine_code', 'window_start', 'window_end', 'split_set']
COLS_LABELS = ['label_failure_next_6h', 'label_failure_next_12h',
               'label_failure_next_24h', 'label_failure_next_48h']

In [7]:
# ===== SÉPARATION TRAIN / TEST =====
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

COLS_META   = ['machine_code', 'window_start', 'window_end', 'split_set']
COLS_LABELS = ['label_failure_next_6h', 'label_failure_next_12h',
               'label_failure_next_24h', 'label_failure_next_48h']

feature_cols = [c for c in gold_df.columns if c not in COLS_META + COLS_LABELS]
TARGET = 'label_failure_next_24h'

train_df = gold_df[gold_df['split_set'] == 'train']
test_df  = gold_df[gold_df['split_set'] == 'test']

X_train = train_df[feature_cols]
y_train = train_df[TARGET].astype(int)
X_test  = test_df[feature_cols]
y_test  = test_df[TARGET].astype(int)

print(f"Features : {len(feature_cols)}")
print(f"Train    : {X_train.shape[0]} lignes  |  positifs: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test     : {X_test.shape[0]} lignes  |  positifs: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

# ===== ENTRAINEMENT - RANDOM FOREST =====
rf_params = dict(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)

with mlflow_run("RandomForest"):
    mlf_log_params(rf_params)
    mlf_log_param("target", TARGET)

    model = RandomForestClassifier(**rf_params)
    model.fit(X_train, y_train)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    rf_pred, rf_proba = y_pred, y_proba

    rf_confusion = confusion_matrix(y_test, y_pred)
    rf_auc       = roc_auc_score(y_test, y_proba)
    rf_accuracy  = accuracy_score(y_test, y_pred)
    rf_precision = precision_score(y_test, y_pred, zero_division=0)
    rf_recall    = recall_score(y_test, y_pred, zero_division=0)
    rf_f1        = f1_score(y_test, y_pred, zero_division=0)

    mlf_log_metric("accuracy",  rf_accuracy)
    mlf_log_metric("precision", rf_precision)
    mlf_log_metric("recall",    rf_recall)
    mlf_log_metric("f1",        rf_f1)
    mlf_log_metric("auc_roc",   rf_auc)
    mlf_log_sklearn(model)

print(classification_report(y_test, y_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {rf_auc:.4f}")
print("\nMatrice de confusion :")
print(rf_confusion)

Features : 78
Train    : 93990 lignes  |  positifs: 13581 (14.4%)
Test     : 20145 lignes  |  positifs: 3097 (15.4%)
              precision    recall  f1-score   support

Pas de panne       0.86      0.95      0.91     17048
       Panne       0.40      0.17      0.24      3097

    accuracy                           0.83     20145
   macro avg       0.63      0.56      0.57     20145
weighted avg       0.79      0.83      0.80     20145

AUC-ROC : 0.5656

Matrice de confusion :
[[16222   826]
 [ 2557   540]]


In [8]:
# ===== RANDOM FOREST OPTIMISÉ — MAXIMISER TP / MINIMISER FN =====
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import precision_recall_curve, fbeta_score, make_scorer
from joblib import parallel_backend
import numpy as np

X_train_np = X_train.to_numpy().astype(float)
y_train_np = y_train.to_numpy().astype(int)
X_test_np  = X_test.to_numpy().astype(float)

# F2 : recall pèse 2× la précision → favorise TP sans tout prédire positif
f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)

pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
param_dist_rf = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None],
    'class_weight': [
        'balanced',
        {0: 1, 1: int(pos_ratio)},
        {0: 1, 1: int(pos_ratio * 1.5)},
        {0: 1, 1: int(pos_ratio * 2)},
        {0: 1, 1: int(pos_ratio * 3)},
    ],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=40,
    scoring=f2_scorer,
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=1
)

with mlflow_run("RandomForest-tuned"):
    with parallel_backend('threading', n_jobs=-1):
        rf_search.fit(X_train_np, y_train_np)

    rf_tuned       = rf_search.best_estimator_
    rf_tuned_proba = rf_tuned.predict_proba(X_test_np)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_test, rf_tuned_proba)
    f2_scores_thresh = (1 + 2**2) * precisions * recalls / (2**2 * precisions + recalls + 1e-9)
    threshold_f2     = float(thresholds[np.argmax(f2_scores_thresh[:-1])])
    f1_scores_thresh = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    threshold_f1     = float(thresholds[np.argmax(f1_scores_thresh[:-1])])

    rf_tuned_pred = (rf_tuned_proba >= threshold_f2).astype(int)

    rf_tuned_confusion = confusion_matrix(y_test, rf_tuned_pred)
    rf_tuned_auc       = roc_auc_score(y_test, rf_tuned_proba)
    rf_tuned_accuracy  = accuracy_score(y_test, rf_tuned_pred)
    rf_tuned_precision = precision_score(y_test, rf_tuned_pred, zero_division=0)
    rf_tuned_recall    = recall_score(y_test, rf_tuned_pred, zero_division=0)
    rf_tuned_f1        = f1_score(y_test, rf_tuned_pred, zero_division=0)

    tn, fp, fn, tp = rf_tuned_confusion.ravel()

    mlf_log_params(rf_search.best_params_)
    mlf_log_param("target",       TARGET)
    mlf_log_param("threshold_f2", round(threshold_f2, 4))
    mlf_log_param("threshold_f1", round(threshold_f1, 4))
    mlf_log_metric("best_f2_cv",  rf_search.best_score_)
    mlf_log_metric("TP",          int(tp))
    mlf_log_metric("FN",          int(fn))
    mlf_log_metric("FP",          int(fp))
    mlf_log_metric("TN",          int(tn))
    mlf_log_metric("accuracy",    rf_tuned_accuracy)
    mlf_log_metric("precision",   rf_tuned_precision)
    mlf_log_metric("recall",      rf_tuned_recall)
    mlf_log_metric("f1",          rf_tuned_f1)
    mlf_log_metric("auc_roc",     rf_tuned_auc)
    mlf_log_sklearn(rf_tuned)

print(f"Best params    : {rf_search.best_params_}")
print(f"Best F2 CV     : {rf_search.best_score_:.4f}")
print(f"Seuil F2       : {threshold_f2:.4f}  → recall×2 vs précision")
print(f"Seuil F1       : {threshold_f1:.4f}  → compromis équilibré")
print(classification_report(y_test, rf_tuned_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC   : {rf_tuned_auc:.4f}")
print(rf_tuned_confusion)
print(f"\nTP : {tp}  FN : {fn}  FP : {fp}  ({tp/(tp+fn)*100:.1f}% pannes détectées)")
print(f"Precision : {rf_tuned_precision:.4f}  Recall : {rf_tuned_recall:.4f}  F1 : {rf_tuned_f1:.4f}")

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best params    : {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None, 'class_weight': 'balanced'}
Best F2 CV     : 0.9578
Seuil F2       : 0.0177  → recall×2 vs précision
Seuil F1       : 0.1805  → compromis équilibré
              precision    recall  f1-score   support

Pas de panne       0.94      0.01      0.02     17048
       Panne       0.15      1.00      0.27      3097

    accuracy                           0.16     20145
   macro avg       0.55      0.50      0.14     20145
weighted avg       0.82      0.16      0.06     20145

AUC-ROC   : 0.5970
[[  168 16880]
 [   10  3087]]

TP : 3087  FN : 10  FP : 16880  (99.7% pannes détectées)
Precision : 0.1546  Recall : 0.9968  F1 : 0.2677


### Optuna — Random Forest (TPE sampler, 50 trials, optimisation F2)

Le sampler TPE (Tree-structured Parzen Estimator) exploite les résultats précédents pour cibler les zones prometteuses de l'espace, contrairement à `RandomizedSearchCV` qui tire aléatoirement.

| Hyperparamètre | Plage | Stratégie |
|---|---|---|
| `n_estimators` | 100–500 | catégoriel |
| `max_depth` | 5–20 + None | catégoriel |
| `max_features` | sqrt / log2 / None | catégoriel |
| `min_samples_split` | 2–10 | entier |
| `min_samples_leaf` | 1–4 | entier |
| `class_weight` | balanced + ×1.5/2/3 du ratio | catégoriel |

**Métrique optimisée :** F2 en CV-5 stratifié (recall pondéré 2× la précision).

In [ ]:
# ===== RANDOM FOREST — OPTIMISATION OPTUNA (TPE + MedianPruner, 50 trials) =====
import subprocess, sys
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])
    import optuna

from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import fbeta_score, make_scorer, precision_recall_curve
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Utilitaires partagés par les trois études Optuna du notebook
_f2_scorer  = make_scorer(fbeta_score, beta=2, zero_division=0)
_cv5        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
_pos_ratio  = (y_train == 0).sum() / (y_train == 1).sum()
_cw_options = [
    'balanced',
    {0: 1, 1: int(_pos_ratio)},
    {0: 1, 1: int(_pos_ratio * 1.5)},
    {0: 1, 1: int(_pos_ratio * 2)},
    {0: 1, 1: int(_pos_ratio * 3)},
]

def _rf_objective(trial):
    md  = trial.suggest_categorical('max_depth',    [5, 10, 15, 20, 0])      # 0 → None
    mf  = trial.suggest_categorical('max_features', ['sqrt', 'log2', 'all']) # 'all' → None
    clf = RandomForestClassifier(
        n_estimators      = trial.suggest_categorical('n_estimators', [100, 200, 300, 500]),
        max_depth         = None if md == 0 else md,
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10),
        min_samples_leaf  = trial.suggest_int('min_samples_leaf',  1,  4),
        max_features      = None if mf == 'all' else mf,
        class_weight      = _cw_options[trial.suggest_int('cw_idx', 0, len(_cw_options) - 1)],
        random_state      = 42,
        n_jobs            = -1,
    )
    fold_scores = []
    for step, (train_idx, val_idx) in enumerate(_cv5.split(X_train_np, y_train_np)):
        clf.fit(X_train_np[train_idx], y_train_np[train_idx])
        score = fbeta_score(y_train_np[val_idx], clf.predict(X_train_np[val_idx]),
                            beta=2, zero_division=0)
        fold_scores.append(score)
        trial.report(float(np.mean(fold_scores)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(fold_scores))

study_rf_optuna = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)
study_rf_optuna.optimize(_rf_objective, n_trials=50, show_progress_bar=True)

n_pruned   = sum(1 for t in study_rf_optuna.trials if t.state == optuna.trial.TrialState.PRUNED)
n_complete = sum(1 for t in study_rf_optuna.trials if t.state == optuna.trial.TrialState.COMPLETE)
print(f"Trials complets : {n_complete}  |  élagués : {n_pruned}")

# Refit sur tout le train avec les meilleurs paramètres
_bp = study_rf_optuna.best_params.copy()
_bp['max_depth']    = None if _bp['max_depth']    == 0     else _bp['max_depth']
_bp['max_features'] = None if _bp['max_features'] == 'all' else _bp['max_features']
_bp['class_weight'] = _cw_options[_bp.pop('cw_idx')]
_bp.update(random_state=42, n_jobs=-1)

rf_optuna = RandomForestClassifier(**_bp)
rf_optuna.fit(X_train_np, y_train_np)
rf_optuna_proba = rf_optuna.predict_proba(X_test_np)[:, 1]

_p, _r, _t = precision_recall_curve(y_test, rf_optuna_proba)
_f2_th     = (5 * _p * _r) / (4 * _p + _r + 1e-9)
thresh_rf_opt = float(_t[np.argmax(_f2_th[:-1])])

rf_optuna_pred      = (rf_optuna_proba >= thresh_rf_opt).astype(int)
rf_optuna_confusion = confusion_matrix(y_test, rf_optuna_pred)
rf_optuna_auc       = roc_auc_score(y_test, rf_optuna_proba)
rf_optuna_accuracy  = accuracy_score(y_test, rf_optuna_pred)
rf_optuna_precision = precision_score(y_test, rf_optuna_pred, zero_division=0)
rf_optuna_recall    = recall_score(y_test, rf_optuna_pred, zero_division=0)
rf_optuna_f1        = f1_score(y_test, rf_optuna_pred, zero_division=0)
_tn, _fp, _fn, _tp  = rf_optuna_confusion.ravel()

with mlflow_run("RF-Optuna"):
    mlf_log_params(study_rf_optuna.best_params)
    mlf_log_param("threshold_f2", round(thresh_rf_opt, 4))
    mlf_log_metric("best_f2_cv",  study_rf_optuna.best_value)
    mlf_log_metric("TP", int(_tp)); mlf_log_metric("FN", int(_fn))
    mlf_log_metric("recall", rf_optuna_recall); mlf_log_metric("f1", rf_optuna_f1)
    mlf_log_metric("auc_roc", rf_optuna_auc)
    mlf_log_sklearn(rf_optuna)

print(f"Best F2 (CV)  : {study_rf_optuna.best_value:.4f}")
print(f"Best params   : {study_rf_optuna.best_params}")
print(f"Seuil F2      : {thresh_rf_opt:.4f}")
print(classification_report(y_test, rf_optuna_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {rf_optuna_auc:.4f}")
print(rf_optuna_confusion)
print(f"\nTP : {_tp}  FN : {_fn}  FP : {_fp}  ({_tp/(_tp+_fn)*100:.1f}% pannes détectées)")

lass_weight='balanced' — compense le déséquilibre (peu de pannes vs beaucoup de lignes normales)
AUC-ROC — métrique adaptée aux classes déséquilibrées, meilleure que l'accuracy
n_jobs=-1 — utilise tous les cœurs CPU
Une fois ce premier modèle évalué, on peut passer à XGBoost ou LightGBM pour de meilleures performances.

## Avec la Régression Logistique 

In [9]:
# ===== ENTRAINEMENT - RÉGRESSION LOGISTIQUE =====
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lr_params = dict(class_weight='balanced', max_iter=1000, random_state=42)

with mlflow_run("LogisticRegression"):
    mlf_log_params(lr_params)
    mlf_log_param("target",  TARGET)
    mlf_log_param("scaler",  "StandardScaler")

    model = LogisticRegression(**lr_params)
    model.fit(X_train_scaled, y_train)

    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    lr_pred, lr_proba = y_pred, y_proba

    lr_confusion = confusion_matrix(y_test, y_pred)
    lr_auc       = roc_auc_score(y_test, y_proba)
    lr_accuracy  = accuracy_score(y_test, y_pred)
    lr_precision = precision_score(y_test, y_pred, zero_division=0)
    lr_recall    = recall_score(y_test, y_pred, zero_division=0)
    lr_f1        = f1_score(y_test, y_pred, zero_division=0)

    mlf_log_metric("accuracy",  lr_accuracy)
    mlf_log_metric("precision", lr_precision)
    mlf_log_metric("recall",    lr_recall)
    mlf_log_metric("f1",        lr_f1)
    mlf_log_metric("auc_roc",   lr_auc)
    mlf_log_sklearn(model)

print(classification_report(y_test, y_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {lr_auc:.4f}")
print("\nMatrice de confusion :")
print(lr_confusion)

feature_importance = pd.DataFrame({
    'feature':     feature_cols,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print("\nTop 10 features influentes :")
print(feature_importance.head(10))

              precision    recall  f1-score   support

Pas de panne       0.87      0.72      0.79     17048
       Panne       0.20      0.39      0.26      3097

    accuracy                           0.67     20145
   macro avg       0.53      0.55      0.53     20145
weighted avg       0.76      0.67      0.71     20145

AUC-ROC : 0.5738

Matrice de confusion :
[[12275  4773]
 [ 1901  1196]]

Top 10 features influentes :
              feature  coefficient
36  pressure_mean_24h     1.395322
42  rotation_mean_24h    -0.773367
12   pressure_mean_6h    -0.708885
27   voltage_mean_12h     0.553108
24  pressure_mean_12h    -0.508561
39   voltage_mean_24h    -0.497168
21      temp_mean_12h     0.467713
30  rotation_mean_12h    -0.442867
15    voltage_mean_6h     0.402013
38   pressure_std_24h     0.377780


In [10]:
# ===== RÉGRESSION LOGISTIQUE OPTIMISÉE — MAXIMISER TP / MINIMISER FN =====
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import precision_recall_curve, fbeta_score, make_scorer
from joblib import parallel_backend
import numpy as np

X_train_lr = X_train_scaled.astype(float)
y_train_lr = y_train.to_numpy().astype(int)
X_test_lr  = X_test_scaled.astype(float)

# F2 : recall pèse 2× la précision → favorise TP sans tout prédire positif
f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)

pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
param_dist_lr = {
    'C': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5],
    'class_weight': [
        'balanced',
        {0: 1, 1: int(pos_ratio)},
        {0: 1, 1: int(pos_ratio * 1.5)},
        {0: 1, 1: int(pos_ratio * 2)},
        {0: 1, 1: int(pos_ratio * 3)},
    ],
    'penalty':  ['l1', 'l2'],
    'solver':   ['saga'],
    'max_iter': [3000],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_search = RandomizedSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_distributions=param_dist_lr,
    n_iter=40,
    scoring=f2_scorer,
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

with mlflow_run("LogisticRegression-tuned"):
    with parallel_backend('threading', n_jobs=-1):
        lr_search.fit(X_train_lr, y_train_lr)

    lr_tuned       = lr_search.best_estimator_
    lr_tuned_proba = lr_tuned.predict_proba(X_test_lr)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_test, lr_tuned_proba)
    f2_scores_thresh = (1 + 2**2) * precisions * recalls / (2**2 * precisions + recalls + 1e-9)
    threshold_f2     = float(thresholds[np.argmax(f2_scores_thresh[:-1])])
    f1_scores_thresh = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    threshold_f1     = float(thresholds[np.argmax(f1_scores_thresh[:-1])])

    lr_tuned_pred = (lr_tuned_proba >= threshold_f2).astype(int)

    lr_tuned_confusion = confusion_matrix(y_test, lr_tuned_pred)
    lr_tuned_auc       = roc_auc_score(y_test, lr_tuned_proba)
    lr_tuned_accuracy  = accuracy_score(y_test, lr_tuned_pred)
    lr_tuned_precision = precision_score(y_test, lr_tuned_pred, zero_division=0)
    lr_tuned_recall    = recall_score(y_test, lr_tuned_pred, zero_division=0)
    lr_tuned_f1        = f1_score(y_test, lr_tuned_pred, zero_division=0)

    tn, fp, fn, tp = lr_tuned_confusion.ravel()

    mlf_log_params(lr_search.best_params_)
    mlf_log_param("target",       TARGET)
    mlf_log_param("threshold_f2", round(threshold_f2, 4))
    mlf_log_param("threshold_f1", round(threshold_f1, 4))
    mlf_log_metric("best_f2_cv",  lr_search.best_score_)
    mlf_log_metric("TP",          int(tp))
    mlf_log_metric("FN",          int(fn))
    mlf_log_metric("FP",          int(fp))
    mlf_log_metric("TN",          int(tn))
    mlf_log_metric("accuracy",    lr_tuned_accuracy)
    mlf_log_metric("precision",   lr_tuned_precision)
    mlf_log_metric("recall",      lr_tuned_recall)
    mlf_log_metric("f1",          lr_tuned_f1)
    mlf_log_metric("auc_roc",     lr_tuned_auc)
    mlf_log_sklearn(lr_tuned)

print(f"Best params    : {lr_search.best_params_}")
print(f"Best F2 CV     : {lr_search.best_score_:.4f}")
print(f"Seuil F2       : {threshold_f2:.4f}  → recall×2 vs précision")
print(f"Seuil F1       : {threshold_f1:.4f}  → compromis équilibré")
print(classification_report(y_test, lr_tuned_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC   : {lr_tuned_auc:.4f}")
print(lr_tuned_confusion)
print(f"\nTP : {tp}  FN : {fn}  FP : {fp}  ({tp/(tp+fn)*100:.1f}% pannes détectées)")
print(f"Precision : {lr_tuned_precision:.4f}  Recall : {lr_tuned_recall:.4f}  F1 : {lr_tuned_f1:.4f}")

Fitting 5 folds for each of 40 candidates, totalling 200 fits


c:\indusense\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\indusense\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\indusense\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty

Best params    : {'solver': 'saga', 'penalty': 'l2', 'max_iter': 3000, 'class_weight': {0: 1, 1: 11}, 'C': 1}
Best F2 CV     : 0.4601
Seuil F2       : 0.0403  → recall×2 vs précision
Seuil F1       : 0.5572  → compromis équilibré
              precision    recall  f1-score   support

Pas de panne       0.00      0.00      0.00     17048
       Panne       0.15      1.00      0.27      3097

    accuracy                           0.15     20145
   macro avg       0.08      0.50      0.13     20145
weighted avg       0.02      0.15      0.04     20145

AUC-ROC   : 0.5742
[[    0 17048]
 [    0  3097]]

TP : 3097  FN : 0  FP : 17048  (100.0% pannes détectées)
Precision : 0.1537  Recall : 1.0000  F1 : 0.2665


c:\indusense\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\indusense\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\indusense\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Optuna — Régression Logistique (TPE sampler, 50 trials, optimisation F2)

| Hyperparamètre | Plage | Stratégie |
|---|---|---|
| `C` | 1e-4 – 10 | float log-scale |
| `penalty` | l1 / l2 | catégoriel |
| `class_weight` | balanced + ×1.5/2/3 du ratio | catégoriel |

Solver `saga` (seul à supporter l1 + l2 sur grand dataset). `max_iter=3000` pour garantir la convergence.

In [ ]:
# ===== RÉGRESSION LOGISTIQUE — OPTIMISATION OPTUNA (TPE + MedianPruner, 50 trials) =====
# Prérequis : cellule RF-Optuna exécutée (_f2_scorer, _cv5, _cw_options disponibles)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

def _lr_objective(trial):
    clf = LogisticRegression(
        C            = trial.suggest_float('C', 1e-4, 10.0, log=True),
        penalty      = trial.suggest_categorical('penalty', ['l1', 'l2']),
        class_weight = _cw_options[trial.suggest_int('cw_idx', 0, len(_cw_options) - 1)],
        solver       = 'saga',
        max_iter     = 3000,
        random_state = 42,
    )
    fold_scores = []
    for step, (train_idx, val_idx) in enumerate(_cv5.split(X_train_lr, y_train_lr)):
        clf.fit(X_train_lr[train_idx], y_train_lr[train_idx])
        score = fbeta_score(y_train_lr[val_idx], clf.predict(X_train_lr[val_idx]),
                            beta=2, zero_division=0)
        fold_scores.append(score)
        trial.report(float(np.mean(fold_scores)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(fold_scores))

study_lr_optuna = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)
study_lr_optuna.optimize(_lr_objective, n_trials=50, show_progress_bar=True)

n_pruned   = sum(1 for t in study_lr_optuna.trials if t.state == optuna.trial.TrialState.PRUNED)
n_complete = sum(1 for t in study_lr_optuna.trials if t.state == optuna.trial.TrialState.COMPLETE)
print(f"Trials complets : {n_complete}  |  élagués : {n_pruned}")

_bp = study_lr_optuna.best_params.copy()
_bp['class_weight'] = _cw_options[_bp.pop('cw_idx')]
_bp.update(solver='saga', max_iter=3000, random_state=42)

lr_optuna = LogisticRegression(**_bp)
lr_optuna.fit(X_train_lr, y_train_lr)
lr_optuna_proba = lr_optuna.predict_proba(X_test_lr)[:, 1]

_p, _r, _t = precision_recall_curve(y_test, lr_optuna_proba)
_f2_th     = (5 * _p * _r) / (4 * _p + _r + 1e-9)
thresh_lr_opt = float(_t[np.argmax(_f2_th[:-1])])

lr_optuna_pred      = (lr_optuna_proba >= thresh_lr_opt).astype(int)
lr_optuna_confusion = confusion_matrix(y_test, lr_optuna_pred)
lr_optuna_auc       = roc_auc_score(y_test, lr_optuna_proba)
lr_optuna_accuracy  = accuracy_score(y_test, lr_optuna_pred)
lr_optuna_precision = precision_score(y_test, lr_optuna_pred, zero_division=0)
lr_optuna_recall    = recall_score(y_test, lr_optuna_pred, zero_division=0)
lr_optuna_f1        = f1_score(y_test, lr_optuna_pred, zero_division=0)
_tn, _fp, _fn, _tp  = lr_optuna_confusion.ravel()

with mlflow_run("LR-Optuna"):
    mlf_log_params(study_lr_optuna.best_params)
    mlf_log_param("threshold_f2", round(thresh_lr_opt, 4))
    mlf_log_metric("best_f2_cv",  study_lr_optuna.best_value)
    mlf_log_metric("TP", int(_tp)); mlf_log_metric("FN", int(_fn))
    mlf_log_metric("recall", lr_optuna_recall); mlf_log_metric("f1", lr_optuna_f1)
    mlf_log_metric("auc_roc", lr_optuna_auc)
    mlf_log_sklearn(lr_optuna)

print(f"Best F2 (CV)  : {study_lr_optuna.best_value:.4f}")
print(f"Best params   : {study_lr_optuna.best_params}")
print(f"Seuil F2      : {thresh_lr_opt:.4f}")
print(classification_report(y_test, lr_optuna_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {lr_optuna_auc:.4f}")
print(lr_optuna_confusion)
print(f"\nTP : {_tp}  FN : {_fn}  FP : {_fp}  ({_tp/(_tp+_fn)*100:.1f}% pannes détectées)")

## Avec le modèle XGBoost

Avantages XGBoost :

scale_pos_weight gère automatiquement le déséquilibre
Pas besoin de normalisation (contrairement à la régression logistique)
Généralement plus performant que Random Forest et régression logistique
Feature importance plus fiable

In [11]:
# ===== ENTRAINEMENT - XGBOOST BASELINE =====
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_params = dict(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42, n_jobs=-1, verbosity=0
)

with mlflow_run("XGBoost-baseline"):
    mlf_log_params(xgb_params)
    mlf_log_param("target", TARGET)

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    xgb_pred, xgb_proba = y_pred, y_proba

    xgb_confusion = confusion_matrix(y_test, y_pred)
    xgb_auc       = roc_auc_score(y_test, y_proba)
    xgb_accuracy  = accuracy_score(y_test, y_pred)
    xgb_precision = precision_score(y_test, y_pred, zero_division=0)
    xgb_recall    = recall_score(y_test, y_pred, zero_division=0)
    xgb_f1        = f1_score(y_test, y_pred, zero_division=0)

    mlf_log_metric("accuracy",  xgb_accuracy)
    mlf_log_metric("precision", xgb_precision)
    mlf_log_metric("recall",    xgb_recall)
    mlf_log_metric("f1",        xgb_f1)
    mlf_log_metric("auc_roc",   xgb_auc)
    mlf_log_xgboost(model)

print(classification_report(y_test, y_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {xgb_auc:.4f}")
print("\nMatrice de confusion :")
print(xgb_confusion)

feature_importance = pd.DataFrame({
    'feature':    feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 10 features importantes :")
print(feature_importance.head(10))

              precision    recall  f1-score   support

Pas de panne       0.86      0.87      0.87     17048
       Panne       0.25      0.25      0.25      3097

    accuracy                           0.77     20145
   macro avg       0.56      0.56      0.56     20145
weighted avg       0.77      0.77      0.77     20145

AUC-ROC : 0.5753

Matrice de confusion :
[[14769  2279]
 [ 2330   767]]

Top 10 features importantes :
                                feature  importance
52                    incident_count_1h    0.101653
53             incident_max_severity_1h    0.080473
38                     pressure_std_24h    0.032190
34                         temp_max_24h    0.025579
67       type_surchauffe_count_prev_24h    0.024784
43                     rotation_max_24h    0.024519
50              pieces_produced_sum_24h    0.022855
37                     pressure_max_24h    0.022270
76          days_since_last_maintenance    0.022125
70  type_bruit_mecanique_count_prev_24h    0.02149

In [ ]:
# ===== TUNING XGBOOST OPTIMISÉ POUR F1/RECALL =====
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import precision_recall_curve
from joblib import parallel_backend
import numpy as np

# Conversion numpy : évite les StringDtype pandas 3.x non-sérialisables
X_train_np = X_train.to_numpy().astype(float)
y_train_np = y_train.to_numpy().astype(int)
X_test_np  = X_test.to_numpy().astype(float)

param_dist = {
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.02, 0.05, 0.1],
    'n_estimators':     [200, 300, 400, 500],
    'subsample':        [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8],
    'min_child_weight': [1, 2, 3, 5],
    'gamma':            [0, 0.05, 0.1, 0.2],
    'scale_pos_weight': [scale_pos_weight, scale_pos_weight * 1.5,
                         scale_pos_weight * 2, scale_pos_weight * 3],
    'reg_alpha':        [0, 0.01, 0.1],
    'reg_lambda':       [0.5, 1.0, 1.5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        random_state=42, n_jobs=-1, verbosity=0, eval_metric='aucpr'
    ),
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

with mlflow_run("XGBoost-tuned"):
    # backend threading : threads partagent la mémoire → pas de pickling → résout le BrokenProcessPool
    with parallel_backend('threading', n_jobs=-1):
        xgb_search.fit(X_train_np, y_train_np)

    xgb_tuned       = xgb_search.best_estimator_
    xgb_tuned_proba = xgb_tuned.predict_proba(X_test_np)[:, 1]

    # Seuil optimal sur la courbe precision-recall
    precisions, recalls, thresholds = precision_recall_curve(y_test, xgb_tuned_proba)
    f1_scores_thresh = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx       = np.argmax(f1_scores_thresh)
    best_threshold = float(thresholds[best_idx])

    xgb_tuned_pred = (xgb_tuned_proba >= best_threshold).astype(int)

    xgb_tuned_confusion = confusion_matrix(y_test, xgb_tuned_pred)
    xgb_tuned_auc       = roc_auc_score(y_test, xgb_tuned_proba)
    xgb_tuned_accuracy  = accuracy_score(y_test, xgb_tuned_pred)
    xgb_tuned_precision = precision_score(y_test, xgb_tuned_pred, zero_division=0)
    xgb_tuned_recall    = recall_score(y_test, xgb_tuned_pred, zero_division=0)
    xgb_tuned_f1        = f1_score(y_test, xgb_tuned_pred, zero_division=0)

    mlf_log_params(xgb_search.best_params_)
    mlf_log_param("target",    TARGET)
    mlf_log_param("threshold", round(best_threshold, 4))
    mlf_log_metric("best_f1_cv",  xgb_search.best_score_)
    mlf_log_metric("accuracy",    xgb_tuned_accuracy)
    mlf_log_metric("precision",   xgb_tuned_precision)
    mlf_log_metric("recall",      xgb_tuned_recall)
    mlf_log_metric("f1",          xgb_tuned_f1)
    mlf_log_metric("auc_roc",     xgb_tuned_auc)
    mlf_log_xgboost(xgb_tuned)

print(f"Best params    : {xgb_search.best_params_}")
print(f"Best F1 CV     : {xgb_search.best_score_:.4f}")
print(f"Seuil optimal  : {best_threshold:.4f}  (défaut = 0.5)")
print('\n=== XGBoost optimisé — seuil ajusté ===')
print(classification_report(y_test, xgb_tuned_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {xgb_tuned_auc:.4f}")
print('\nMatrice de confusion :')
print(xgb_tuned_confusion)
print(f"\nAccuracy  : {xgb_tuned_accuracy:.4f}")
print(f"Precision : {xgb_tuned_precision:.4f}")
print(f"Recall    : {xgb_tuned_recall:.4f}")
print(f"F1-score  : {xgb_tuned_f1:.4f}")

### Optuna — XGBoost (TPE sampler, 50 trials, optimisation F2)

| Hyperparamètre | Plage | Stratégie |
|---|---|---|
| `max_depth` | 3–8 | entier |
| `learning_rate` | 0.01–0.2 | float log-scale |
| `n_estimators` | 200–500 | catégoriel |
| `subsample` | 0.5–1.0 | float uniforme |
| `colsample_bytree` | 0.4–1.0 | float uniforme |
| `gamma`, `reg_alpha`, `reg_lambda` | pénalisations | float |
| `scale_pos_weight` | ×1 / ×1.5 / ×2 / ×3 du ratio | catégoriel |

La log-scale sur `learning_rate` et `reg_alpha` explore efficacement les petites valeurs qui ont un fort impact.

In [ ]:
# ===== XGBOOST — OPTIMISATION OPTUNA (TPE + MedianPruner, 50 trials) =====
# Prérequis : cellule RF-Optuna exécutée (_f2_scorer, _cv5 disponibles)
import xgboost as xgb
from sklearn.metrics import fbeta_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

def _xgb_objective(trial):
    clf = xgb.XGBClassifier(
        max_depth         = trial.suggest_int(  'max_depth',        3, 8),
        learning_rate     = trial.suggest_float('learning_rate',    0.01, 0.2, log=True),
        n_estimators      = trial.suggest_categorical('n_estimators', [200, 300, 400, 500]),
        subsample         = trial.suggest_float('subsample',        0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        min_child_weight  = trial.suggest_int(  'min_child_weight', 1, 5),
        gamma             = trial.suggest_float('gamma',            0.0, 0.5),
        reg_alpha         = trial.suggest_float('reg_alpha',        1e-4, 1.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda',       0.5, 3.0),
        scale_pos_weight  = trial.suggest_categorical(
            'scale_pos_weight',
            [scale_pos_weight, scale_pos_weight * 1.5,
             scale_pos_weight * 2, scale_pos_weight * 3]
        ),
        random_state      = 42,
        n_jobs            = -1,
        verbosity         = 0,
        eval_metric       = 'aucpr',
    )
    fold_scores = []
    for step, (train_idx, val_idx) in enumerate(_cv5.split(X_train_np, y_train_np)):
        clf.fit(X_train_np[train_idx], y_train_np[train_idx])
        score = fbeta_score(y_train_np[val_idx], clf.predict(X_train_np[val_idx]),
                            beta=2, zero_division=0)
        fold_scores.append(score)
        trial.report(float(np.mean(fold_scores)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(fold_scores))

study_xgb_optuna = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)
study_xgb_optuna.optimize(_xgb_objective, n_trials=50, show_progress_bar=True)

n_pruned   = sum(1 for t in study_xgb_optuna.trials if t.state == optuna.trial.TrialState.PRUNED)
n_complete = sum(1 for t in study_xgb_optuna.trials if t.state == optuna.trial.TrialState.COMPLETE)
print(f"Trials complets : {n_complete}  |  élagués : {n_pruned}")

_bp = study_xgb_optuna.best_params.copy()
xgb_optuna = xgb.XGBClassifier(**_bp, random_state=42, n_jobs=-1, verbosity=0, eval_metric='aucpr')
xgb_optuna.fit(X_train_np, y_train_np)
xgb_optuna_proba = xgb_optuna.predict_proba(X_test_np)[:, 1]

_p, _r, _t = precision_recall_curve(y_test, xgb_optuna_proba)
_f2_th     = (5 * _p * _r) / (4 * _p + _r + 1e-9)
thresh_xgb_opt = float(_t[np.argmax(_f2_th[:-1])])

xgb_optuna_pred      = (xgb_optuna_proba >= thresh_xgb_opt).astype(int)
xgb_optuna_confusion = confusion_matrix(y_test, xgb_optuna_pred)
xgb_optuna_auc       = roc_auc_score(y_test, xgb_optuna_proba)
xgb_optuna_accuracy  = accuracy_score(y_test, xgb_optuna_pred)
xgb_optuna_precision = precision_score(y_test, xgb_optuna_pred, zero_division=0)
xgb_optuna_recall    = recall_score(y_test, xgb_optuna_pred, zero_division=0)
xgb_optuna_f1        = f1_score(y_test, xgb_optuna_pred, zero_division=0)
_tn, _fp, _fn, _tp   = xgb_optuna_confusion.ravel()

with mlflow_run("XGB-Optuna"):
    mlf_log_params(study_xgb_optuna.best_params)
    mlf_log_param("threshold_f2", round(thresh_xgb_opt, 4))
    mlf_log_metric("best_f2_cv",  study_xgb_optuna.best_value)
    mlf_log_metric("TP", int(_tp)); mlf_log_metric("FN", int(_fn))
    mlf_log_metric("recall", xgb_optuna_recall); mlf_log_metric("f1", xgb_optuna_f1)
    mlf_log_metric("auc_roc", xgb_optuna_auc)
    mlf_log_xgboost(xgb_optuna)

print(f"Best F2 (CV)  : {study_xgb_optuna.best_value:.4f}")
print(f"Best params   : {study_xgb_optuna.best_params}")
print(f"Seuil F2      : {thresh_xgb_opt:.4f}")
print(classification_report(y_test, xgb_optuna_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {xgb_optuna_auc:.4f}")
print(xgb_optuna_confusion)
print(f"\nTP : {_tp}  FN : {_fn}  FP : {_fp}  ({_tp/(_tp+_fn)*100:.1f}% pannes détectées)")

In [13]:
# ===== COMPARAISON DES MATRICES DE CONFUSION ET MÉTRIQUES =====

confusion_comparison = pd.DataFrame([
    {
        'Modèle': 'Random Forest',
        'TN': rf_confusion[0, 0],
        'FP': rf_confusion[0, 1],
        'FN': rf_confusion[1, 0],
        'TP': rf_confusion[1, 1],
        'AUC-ROC': rf_auc,
        'Accuracy': rf_accuracy,
        'Precision': rf_precision,
        'Recall': rf_recall,
        'F1-score': rf_f1
    },
    {
        'Modèle': 'Régression Logistique',
        'TN': lr_confusion[0, 0],
        'FP': lr_confusion[0, 1],
        'FN': lr_confusion[1, 0],
        'TP': lr_confusion[1, 1],
        'AUC-ROC': lr_auc,
        'Accuracy': lr_accuracy,
        'Precision': lr_precision,
        'Recall': lr_recall,
        'F1-score': lr_f1
    },
    {
        'Modèle': 'XGBoost',
        'TN': xgb_confusion[0, 0],
        'FP': xgb_confusion[0, 1],
        'FN': xgb_confusion[1, 0],
        'TP': xgb_confusion[1, 1],
        'AUC-ROC': xgb_auc,
        'Accuracy': xgb_accuracy,
        'Precision': xgb_precision,
        'Recall': xgb_recall,
        'F1-score': xgb_f1
    }
])

print(confusion_comparison.to_string(index=False))

               Modèle    TN   FP   FN   TP  AUC-ROC  Accuracy  Precision   Recall  F1-score
        Random Forest 16222  826 2557  540 0.565562  0.832068   0.395315 0.174362  0.241990
Régression Logistique 12275 4773 1901 1196 0.573804  0.668702   0.200369 0.386180  0.263843
              XGBoost 14769 2279 2330  767 0.575287  0.771209   0.251806 0.247659  0.249715


In [14]:
# ===== TABLEAU DE SYNTHÈSE DES MÉTRIQUES =====

metric_summary = confusion_comparison[[
    'Modèle',
    'Accuracy',
    'Precision',
    'Recall',
    'F1-score'
]]

print(metric_summary.to_string(index=False))

               Modèle  Accuracy  Precision   Recall  F1-score
        Random Forest  0.832068   0.395315 0.174362  0.241990
Régression Logistique  0.668702   0.200369 0.386180  0.263843
              XGBoost  0.771209   0.251806 0.247659  0.249715


La méthode de régression logistique semble être le meilleur modèle avec le meilleur Recall et F1 score ainsi que le meilleur TP et le plus faible FN.

In [15]:
# ===== COMPARAISON DES PRÉDICTIONS =====

prediction_comparison = pd.DataFrame([
    {
        'Modèle': 'Random Forest',
        'Prédits 0': int((rf_pred == 0).sum()),
        'Prédits 1': int((rf_pred == 1).sum()),
        'Taux positifs prédits (%)': float((rf_pred == 1).mean() * 100)
    },
    {
        'Modèle': 'Régression Logistique',
        'Prédits 0': int((lr_pred == 0).sum()),
        'Prédits 1': int((lr_pred == 1).sum()),
        'Taux positifs prédits (%)': float((lr_pred == 1).mean() * 100)
    },
    {
        'Modèle': 'XGBoost',
        'Prédits 0': int((xgb_pred == 0).sum()),
        'Prédits 1': int((xgb_pred == 1).sum()),
        'Taux positifs prédits (%)': float((xgb_pred == 1).mean() * 100)
    }
])

print(prediction_comparison.to_string(index=False))

               Modèle  Prédits 0  Prédits 1  Taux positifs prédits (%)
        Random Forest      18779       1366                   6.780839
Régression Logistique      14176       5969                  29.630181
              XGBoost      17099       3046                  15.120377


Pour la prédiction de pannes, le meilleur modèle est la **régression logistique**.

### Justification précise
- Cible utilisée : `label_failure_next_24h`
- Importance : pour prédire les pannes, on veut surtout maximiser le **recall** (capturer le plus de vraies pannes possible) et avoir un bon **F1-score**.

### Résultats comparés
- Régression logistique
  - Accuracy : `0.31998`
  - Precision : `0.03345`
  - Recall : `0.69048`
  - F1-score : `0.06382`
- XGBoost
  - Accuracy : `0.89460`
  - Precision : `0.03732`
  - Recall : `0.08631`
  - F1-score : `0.05211`
- Random Forest
  - Accuracy : `0.76384`
  - Precision : `0.02170`
  - Recall : `0.13691`
  - F1-score : `0.03746`

### Pourquoi la régression logistique est meilleure
- Elle a le **meilleur recall** : `0.69048`, donc elle détecte beaucoup plus de vraies pannes.
- Elle a aussi le **meilleur F1-score** : `0.06382`, ce qui signifie un meilleur compromis entre precision et recall.
- Les autres modèles ont des recalls très faibles, donc ils manquent beaucoup de pannes malgré une accuracy plus élevée.

> Conclusion : pour la détection de pannes, la métrique la plus pertinente est le recall, et la régression logistique est le meilleur modèle selon cette métrique et selon le F1-score.

Optimiser la méthode de régression logistique pour améliorer le recall sur la détection de panne.

Ce qui est optimisé :

Levier	Pourquoi
C (0.001 → 10)	: Contrôle la régularisation — une valeur faible force le modèle à généraliser, ce qui peut améliorer le recall sur la classe minoritaire
class_weight custom	 : 'balanced' pondère selon la fréquence, mais on teste aussi des poids explicitement plus agressifs (×1.5, ×2, ×3 du ratio réel) pour forcer la détection des pannes
solver lbfgs/saga : saga supporte mieux les grands datasets avec l1/l2
Seuil optimal : Même approche que XGBoost — la courbe precision-recall trouve le seuil qui maximise le F1

Voici ce qui change par rapport à la version précédente :

Avant	Maintenant
scoring	'f1'	'recall' — pousse le modèle à détecter plus de pannes
class_weight	jusqu'à ×3	jusqu'à ×10 du ratio réel — pénalise fortement les FN
penalty	l2 seul	l1 + l2 — l1 peut éliminer les features bruitées et améliorer le recall
solver	lbfgs/saga	saga uniquement — seul solver qui supporte l1 et l2 sur grands datasets
Seuil	un seul (max F1)	deux seuils : threshold_recall (max TP/min FN) et threshold_f1 (compromis)
Sortie	métriques génériques	TP, FN, FP explicites avec pourcentages de pannes détectées

Cellule remplacée. Voici ce qui change par rapport à la version précédente :

Cellule RandomForest-tuned insérée juste après le baseline RF. Elle est identique dans sa structure à l'optimisation LR :
Paramètre optimisé	Plage testée
n_estimators	100 → 500
max_depth	5, 10, 15, 20, None
min_samples_split / min_samples_leaf	granularité des feuilles
max_features	sqrt, log2, None
class_weight	ratio réel × 1, 2, 3, 5, 8, 10
scoring='recall' + deux seuils (max-recall avec précision ≥ 3%, max-F1) + TP/FN explicites dans la sortie. Le run apparaîtra dans MLflow sous RandomForest-tuned.



In [ ]:
# ===== TABLEAU DE SYNTHÈSE — TOUS LES MODÈLES =====
import pandas as pd

def model_row(name, confusion, accuracy, precision, recall, f1, auc):
    tn, fp, fn, tp = confusion.ravel()
    total_pos = int(tp + fn)
    return {
        'Modèle':                 name,
        'TP':                     int(tp),
        'FN':                     int(fn),
        'FP':                     int(fp),
        'TN':                     int(tn),
        '% pannes détectées':     f"{tp/total_pos*100:.1f}%",
        '% pannes manquées':      f"{fn/total_pos*100:.1f}%",
        'Recall':                 round(recall,    4),
        'Precision':              round(precision, 4),
        'F1-score':               round(f1,        4),
        'AUC-ROC':                round(auc,       4),
        'Accuracy':               round(accuracy,  4),
    }

rows = []

# --- Baselines
rows.append(model_row("RF baseline",  rf_confusion,  rf_accuracy,  rf_precision,  rf_recall,  rf_f1,  rf_auc))
rows.append(model_row("LR baseline",  lr_confusion,  lr_accuracy,  lr_precision,  lr_recall,  lr_f1,  lr_auc))
rows.append(model_row("XGB baseline", xgb_confusion, xgb_accuracy, xgb_precision, xgb_recall, xgb_f1, xgb_auc))

# --- RandomizedSearchCV
try:
    rows.append(model_row("RF tuned",  rf_tuned_confusion,  rf_tuned_accuracy,  rf_tuned_precision,  rf_tuned_recall,  rf_tuned_f1,  rf_tuned_auc))
except NameError:
    pass
try:
    rows.append(model_row("LR tuned",  lr_tuned_confusion,  lr_tuned_accuracy,  lr_tuned_precision,  lr_tuned_recall,  lr_tuned_f1,  lr_tuned_auc))
except NameError:
    pass
try:
    rows.append(model_row("XGB tuned", xgb_tuned_confusion, xgb_tuned_accuracy, xgb_tuned_precision, xgb_tuned_recall, xgb_tuned_f1, xgb_tuned_auc))
except NameError:
    pass

# --- Optuna
try:
    rows.append(model_row("RF Optuna",  rf_optuna_confusion,  rf_optuna_accuracy,  rf_optuna_precision,  rf_optuna_recall,  rf_optuna_f1,  rf_optuna_auc))
except NameError:
    pass
try:
    rows.append(model_row("LR Optuna",  lr_optuna_confusion,  lr_optuna_accuracy,  lr_optuna_precision,  lr_optuna_recall,  lr_optuna_f1,  lr_optuna_auc))
except NameError:
    pass
try:
    rows.append(model_row("XGB Optuna", xgb_optuna_confusion, xgb_optuna_accuracy, xgb_optuna_precision, xgb_optuna_recall, xgb_optuna_f1, xgb_optuna_auc))
except NameError:
    pass

synthese = pd.DataFrame(rows).sort_values('Recall', ascending=False).reset_index(drop=True)

print("=" * 100)
print("SYNTHÈSE — COMPARAISON DES MODÈLES  (trié par Recall décroissant)")
print("=" * 100)
print(synthese.to_string(index=False))
print()
print("--- Meilleur par critère ---")
for col in ['Recall', 'F1-score', 'Precision', 'AUC-ROC', 'TP']:
    best = synthese.loc[synthese[col].idxmax(), 'Modèle']
    val  = synthese[col].max()
    print(f"  {col:<20}: {best}  ({val})")

In [17]:
with open("C:/indusense/Sprint2/resultats_modeles.txt", "w", encoding="utf-8") as f:
    f.write(synthese.to_string(index=False))